# Exercise04: Ride On Reimagined Transit Equity Project Phase One

Emmanuel Opoku-Boateng  
URSP688Y Urban Data Science & Smart Cities

In this notebook, I operationalize the GTFS service-change portion of the analysis. The larger project asks whether Ride On Reimagined produced larger scheduled service gains in higher-transit-need neighborhoods than in lower-transit-need neighborhoods.

First step: measuring scheduled Ride On service before and after the first major implementation feed.

## 1. Question For This Portion Of The Analysis

**Question:** How did scheduled Ride On service at the stop level change between the pre-implementation January 2025 GTFS feed and the first post-implementation June 2025 GTFS feed?

This prepares the service-side dataset that will later be joined to Census tracts and ACS-based transit need indicators.

**Note:** GTFS measures scheduled service. It does not measure actual rider experience, reliability, missed trips, crowding, safety, fare burden, or whether transit connects riders to meaningful destinations.

## 2. Overall Approach And Pseudocode

The logic of this notebook is:

1. Choose two official Ride On GTFS feeds that represent the comparison period.
2. Use the GTFS calendar tables to identify a representative weekday and weekend day in each feed.
3. Join GTFS `trips`, `stop_times`, and `stops` so each scheduled stop event can be assigned to a stop.
4. Calculate stop-level service indicators:
   - weekday scheduled trips
   - Saturday scheduled trips
   - weekday peak trips per hour
   - weekday span of service
   - Saturday span of service
5. Compare stop IDs across the pre and post feeds.
6. Save the outputs so the final project can spatially aggregate these stop-level metrics to Census tracts.


## 3. Load Python Tools

1. Import general-purpose libraries for file paths, tables, and plotting.
2. Import the helper functions that convert raw GTFS tables into service metrics.
3. Keep the notebook readable by placing repeated GTFS logic in `gtfs_service_metrics.py`.

In [ ]:

from pathlib import Path

import pandas as pd

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt

# Import reusable GTFS functions from the local project module.
from gtfs_service_metrics import build_feed_metrics, compare_feed_metrics

# Show more table columns in notebook output so the service metrics are inspectable.
pd.set_option("display.max_columns", 80)

# Use a clean plotting style so the chart is readable without extra design work.
plt.style.use("seaborn-v0_8-whitegrid")

## 4. Define Project Paths And Comparison Feeds


1. Define raw, processed, and figure folders relative to that project folder.
2. Select the January 2025 feed as the primary pre-implementation baseline.
3. Select the June 2025 feed as the first post-implementation feed because it begins with the June 29, 2025 schedule period.
4. Check that the raw GTFS ZIP files exist before running the analysis.

In [ ]:
project_dir = Path.cwd()

# Store official downloaded GTFS ZIP files in a raw GTFS subfolder.
raw_gtfs_dir = project_dir / "data" / "raw" / "gtfs"

# Store cleaned service metrics in a processed GTFS subfolder.
processed_gtfs_dir = project_dir / "data" / "processed" / "gtfs"

# Store plots in the figures folder.
figures_dir = project_dir / "figures"

# Make sure output folders exist before saving CSVs or figures.
processed_gtfs_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

# Define the pre-implementation feed label used throughout the analysis.
pre_label = "2025_january"

# Define the post-implementation feed label used throughout the analysis.
post_label = "2025_june"

# Point to the official January 2025 Ride On GTFS archive stored locally.
pre_feed_path = raw_gtfs_dir / "rideon_2025_january.zip"

# Point to the official June 2025 Ride On GTFS archive stored locally.
post_feed_path = raw_gtfs_dir / "rideon_2025_june.zip"

# Stop early with a clear message if the pre feed is missing.
assert pre_feed_path.exists(), f"Missing pre feed: {pre_feed_path}"

# Stop early with a clear message if the post feed is missing.
assert post_feed_path.exists(), f"Missing post feed: {post_feed_path}"

## 5. Inspect The GTFS Feed Inventory


1. Load the prepared GTFS inventory table.
2. Review feed start and end dates to confirm that the selected comparison feeds make sense.
3. Keep the inventory visible because it documents why these two files were selected.

In [3]:
# Point to the GTFS inventory created by the project data preparation script.
inventory_path = processed_gtfs_dir / "gtfs_feed_inventory.csv"

# Load the inventory so we can inspect the schedule periods available for the project.
gtfs_inventory = pd.read_csv(inventory_path)

# Select the columns that explain each feed's role and valid schedule dates.
inventory_view = gtfs_inventory[
    [
        "feed_label",
        "role",
        "feed_start_date",
        "feed_end_date",
        "representative_tuesday",
        "representative_saturday",
        "trips_rows",
        "stops_rows",
    ]
]

# Display the inventory table for documentation and sanity checking.
inventory_view

,feed_label,role,feed_start_date,feed_end_date,representative_tuesday,representative_saturday,trips_rows,stops_rows
0,2024_january,early_pre_implementation_context,20240114,20240504,2024-01-16,2024-01-20,17046,4912
1,2024_may,pre_implementation_context,20240505,20240907,2024-05-07,2024-05-11,13319,4915
2,2024_september,pre_implementation_context,20240908,20250111,2024-09-10,2024-09-14,17716,4939
3,2025_january,primary_pre_phase1_baseline,20250112,20250628,2025-01-14,2025-01-18,17373,4933
4,2025_june,primary_post_phase1_feed,20250629,20250906,2025-07-01,2025-07-05,9745,4599
5,2025_september,post_phase1_followup,20250907,20260110,2025-09-09,2025-09-13,17501,4590
6,2026_may_current,published_current_feed_as_of_2026_04_30,20260503,20260905,2026-05-05,2026-05-09,17530,4578


## 6. Build Stop-Level Service Metrics For Each Feed


1. For the pre feed, find representative Tuesday and Saturday dates from the GTFS calendar.
2. Count scheduled stop-level service for those representative dates.
3. Repeat the same process for the post feed.
4. Save each stop-level metric table so later notebooks can reuse the results without recalculating everything.

The helper function calculates the same metrics for each feed, which makes the comparison reproducible.

In [4]:
# Build stop-level scheduled service metrics for the January 2025 baseline feed.
pre_metrics = build_feed_metrics(pre_feed_path, pre_label)

# Build stop-level scheduled service metrics for the June 2025 post-implementation feed.
post_metrics = build_feed_metrics(post_feed_path, post_label)

# Save the pre metrics to a processed CSV for later spatial analysis.
pre_metrics.to_csv(processed_gtfs_dir / f"stop_service_metrics_{pre_label}.csv", index=False)

# Save the post metrics to a processed CSV for later spatial analysis.
post_metrics.to_csv(processed_gtfs_dir / f"stop_service_metrics_{post_label}.csv", index=False)

# Show the number of served stops represented in each feed.
pd.DataFrame(
    {
        "feed": [pre_label, post_label],
        "served_stops": [len(pre_metrics), len(post_metrics)],
        "representative_weekday": [
            pre_metrics["weekday_analysis_date"].iloc[0],
            post_metrics["weekday_analysis_date"].iloc[0],
        ],
        "representative_saturday": [
            pre_metrics["weekend_analysis_date"].iloc[0],
            post_metrics["weekend_analysis_date"].iloc[0],
        ],
    }
)

,feed,served_stops,representative_weekday,representative_saturday
0,2025_january,4795,2025-01-14,2025-01-18
1,2025_june,4597,2025-07-01,2025-07-05


## 7. Preview The Stop-Level Metrics


1. Sort stops by weekday scheduled trips.
2. Preview the busiest stops in the pre-implementation feed.
3. Check whether the metrics have plausible values before comparing feeds.

In [5]:
# Select columns that are most useful for a first inspection of stop-level service.
preview_columns = [
    "feed_label",
    "stop_id",
    "stop_name",
    "weekday_total_trips",
    "weekend_total_trips",
    "weekday_peak_trips_per_hour",
    "weekday_span_hours",
]

# Display the top 10 stops by scheduled weekday trips in the pre feed.
pre_metrics[preview_columns].head(10)

,feed_label,stop_id,stop_name,weekday_total_trips,weekend_total_trips,weekday_peak_trips_per_hour,weekday_span_hours
0,2025_january,3722,LAKEFOREST TRANSIT CENTER & ODENDHAL AVE,607,459.0,39.00,21.267
1,2025_january,15182,GERMANTOWN TRANSIT CENTER & BAY B,574,406.0,35.50,21.300
2,2025_january,5036,POWDER MILL RD & NEW HAMPSHIRE AVE,281,158.0,19.50,20.783
3,2025_january,6008,SHADY GROVE STATION & BAY E - EAST,268,132.0,17.25,19.467
4,2025_january,14776,SHADY GROVE STATION & BAY A - WEST,250,133.0,16.75,20.600
5,2025_january,15324,S CAMPUS DR & CAMPUS DR,245,199.0,14.50,21.033
6,2025_january,1094,COLESVILLE RD & FENTON ST,236,161.0,16.25,19.967
7,2025_january,9182,GLENMONT STATION & BAY C,232,117.0,19.25,19.967
8,2025_january,4644,ODENDHAL AVE & RUSSELL AVE,232,179.0,14.00,21.192
9,2025_january,4648,ODENDHAL AVE & RUSSELL AVE,232,178.0,13.50,20.245


## 8. Compare Stop-Level Service Before And After Phase 1


1. Outer join the pre and post metrics by `stop_id`.
2. Keep stops that appear in only one feed so as to be able to see additions and removals.
3. Calculate changes in weekday trips, Saturday trips, peak trips per hour, and span of service.
4. Save the comparison table for later spatial aggregation.

In [6]:
# Compare pre and post stop-level metrics using stable GTFS stop IDs.
comparison = compare_feed_metrics(pre_metrics, post_metrics, pre_label, post_label)

# Create one readable stop name by preferring the post-feed name and falling back to the pre-feed name.
comparison["stop_name"] = comparison[f"stop_name_{post_label}"].fillna(comparison[f"stop_name_{pre_label}"])

# Save the stop-level service change table for later tract aggregation.
comparison.to_csv(
    processed_gtfs_dir / f"stop_service_change_{pre_label}_to_{post_label}.csv",
    index=False,
)

# Display the size of the comparison table and the merge categories.
pd.DataFrame(
    {
        "comparison_rows": [len(comparison)],
        "stops_in_both_feeds": [(comparison["_merge"] == "both").sum()],
        "stops_only_in_pre_feed": [(comparison["_merge"] == "left_only").sum()],
        "stops_only_in_post_feed": [(comparison["_merge"] == "right_only").sum()],
    }
)

,comparison_rows,stops_in_both_feeds,stops_only_in_pre_feed,stops_only_in_post_feed
0,4854,4538,257,59


## 9. Summarize The Direction Of Service Change


1. Count how many stops gained weekday scheduled trips.
2. Count how many stops lost weekday scheduled trips.
3. Count how many stops had no weekday scheduled trip change.
4. Summarize the average, median, minimum, and maximum changes.

This is only a descriptive first pass, not a final equity conclusion.

In [7]:
# Store the main change variable in a shorter name for readability.
weekday_change = comparison["weekday_total_trips_change"]

# Count stops with more weekday trips after the implementation feed.
stops_with_gains = (weekday_change > 0).sum()

# Count stops with fewer weekday trips after the implementation feed.
stops_with_losses = (weekday_change < 0).sum()

# Count stops with the same number of weekday trips in both feeds.
stops_with_no_change = (weekday_change == 0).sum()

# Build a compact descriptive summary table.
change_summary = pd.DataFrame(
    {
        "measure": [
            "stops with weekday trip gains",
            "stops with weekday trip losses",
            "stops with no weekday trip change",
            "mean weekday trip change",
            "median weekday trip change",
            "minimum weekday trip change",
            "maximum weekday trip change",
        ],
        "value": [
            stops_with_gains,
            stops_with_losses,
            stops_with_no_change,
            weekday_change.mean(),
            weekday_change.median(),
            weekday_change.min(),
            weekday_change.max(),
        ],
    }
)

# Display the summary table.
change_summary

,measure,value
0,stops with weekday trip gains,204.000000
1,stops with weekday trip losses,353.000000
2,stops with no weekday trip change,4297.000000
3,mean weekday trip change,0.010919
4,median weekday trip change,0.000000
5,minimum weekday trip change,-150.000000
6,maximum weekday trip change,183.000000


## 10. Identify Stops With The Largest Gains And Losses


1. Sort the comparison table by weekday trip change from largest to smallest.
2. Display the stops with the largest scheduled service gains.
3. Sort the same table from smallest to largest.
4. Display the stops with the largest scheduled service losses.

In [8]:
# Select columns that explain stop-level changes clearly.
change_columns = [
    "stop_id",
    "stop_name",
    f"weekday_total_trips_{pre_label}",
    f"weekday_total_trips_{post_label}",
    "weekday_total_trips_change",
    f"weekend_total_trips_{pre_label}",
    f"weekend_total_trips_{post_label}",
    "weekend_total_trips_change",
    "_merge",
]

# Create a table of the 10 largest weekday scheduled trip gains.
largest_gains = comparison.sort_values("weekday_total_trips_change", ascending=False)[change_columns].head(10)

# Create a table of the 10 largest weekday scheduled trip losses.
largest_losses = comparison.sort_values("weekday_total_trips_change", ascending=True)[change_columns].head(10)

# Display the largest gains first.
largest_gains

,stop_id,stop_name,weekday_total_trips_2025_january,weekday_total_trips_2025_june,weekday_total_trips_change,weekend_total_trips_2025_january,weekend_total_trips_2025_june,weekend_total_trips_change,_merge
3701,7184,WHEATON STATION & BAY K,56.0,239.0,183.0,36.0,175.0,139.0,both
525,15324,S CAMPUS DR & CAMPUS DR,245.0,403.0,158.0,199.0,314.0,115.0,both
2930,5654,ROCKVILLE STATION & BAY A - WEST,106.0,210.0,104.0,82.0,163.0,81.0,both
316,14637,VEIRS MILL RD & ASPEN HILL RD,46.0,138.0,92.0,35.0,105.0,70.0,both
318,14639,VEIRS MILL RD & MEADOW HALL DR,NaN,92.0,92.0,NaN,70.0,70.0,right_only
324,14644,VEIRS MILL RD & BROADWOOD DR,NaN,92.0,92.0,NaN,70.0,70.0,right_only
328,14648,VEIRS MILL RD & FIRST ST,NaN,92.0,92.0,NaN,70.0,70.0,right_only
320,14640,VEIRS MILL RD & ATLANTIC AVE,NaN,92.0,92.0,NaN,70.0,70.0,right_only
322,14642,VEIRS MILL RD & NIMITZ AVE,NaN,92.0,92.0,NaN,70.0,70.0,right_only
317,14638,VEIRS MILL RD & TWINBROOK PKWY,NaN,92.0,92.0,NaN,70.0,70.0,right_only


In [9]:
# Display the largest losses after reviewing the gains.
largest_losses

,stop_id,stop_name,weekday_total_trips_2025_january,weekday_total_trips_2025_june,weekday_total_trips_change,weekend_total_trips_2025_january,weekend_total_trips_2025_june,weekend_total_trips_change,_merge
1868,3327,HUNGERFORD DR & @700,150.0,NaN,-150.0,118.0,NaN,-118.0,left_only
2111,3838,MANNAKEE ST & CAMPUS DR,139.0,NaN,-139.0,117.0,NaN,-117.0,left_only
1301,2488,FREDERICK RD & SHADY GROVE RD,121.0,NaN,-121.0,92.0,NaN,-92.0,left_only
1106,2042,EXECUTIVE BLVD & OLD GEORGETOWN RD,101.0,NaN,-101.0,65.0,NaN,-65.0,left_only
1515,300,BETHESDA STATION & BAY A,78.0,NaN,-78.0,58.0,NaN,-58.0,left_only
2907,5608,ROCKVILLE PIKE & OLD GEORGETOWN RD,77.0,NaN,-77.0,59.0,NaN,-59.0,left_only
1453,2862,GEORGIA AVE & VEIRS MILL RD,73.0,NaN,-73.0,43.0,NaN,-43.0,left_only
1421,2790,GEORGIA AVE & RANDOLPH RD,72.0,NaN,-72.0,46.0,NaN,-46.0,left_only
2663,5036,POWDER MILL RD & NEW HAMPSHIRE AVE,281.0,209.0,-72.0,158.0,112.0,-46.0,both
3105,5982,SHADY GROVE RD & CRABBS BRANCH WAY,71.0,NaN,-71.0,49.0,NaN,-49.0,left_only


## 11. Plot The Direction Of Weekday Trip Changes

1. Categorize each stop as a weekday service gain, loss, or no change.
2. Count how many stops fall into each category.
3. Draw a bar chart so the direction of change is easy to read.
4. Save the figure 

In [ ]:
# Categorize each stop based on the sign of its weekday trip change.
comparison["weekday_change_category"] = "No change"

# Label stops with positive change values as service gains.
comparison.loc[comparison["weekday_total_trips_change"] > 0, "weekday_change_category"] = "Gain"

# Label stops with negative change values as service losses.
comparison.loc[comparison["weekday_total_trips_change"] < 0, "weekday_change_category"] = "Loss"

# Count categories in a fixed order so the chart reads from loss to no change to gain.
category_counts = comparison["weekday_change_category"].value_counts().reindex(["Loss", "No change", "Gain"])

# Create a figure and axis for the bar chart.
fig, ax = plt.subplots(figsize=(8, 5))

# Plot the number of stops in each weekday service-change category.
ax.bar(category_counts.index, category_counts.values, color=["#B85C5C", "#7A7A7A", "#2F6B9A"])

# Label the y-axis with the count represented by each bar.
ax.set_ylabel("Number of stops")

# Give the chart a descriptive title.
ax.set_title("Direction of Ride On weekday service change: January 2025 to June 2025")

# Add count labels above each bar for easier interpretation.
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{int(bar.get_height()):,}", ha="center", va="bottom")

# Save the figure to the figures folder for later reuse.
fig.savefig(figures_dir / "weekday_trip_change_direction_2025_jan_to_2025_june.png", dpi=200, bbox_inches="tight")


C:\Users\boatf\AppData\Local\Temp\ipykernel_6620\1194008578.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12. Preliminary Interpretation

In this notebook I create the first required building block for the project: a reproducible stop-level measurement of scheduled Ride On service change.

At this stage, a stop-level gain or loss does not yet tell us whether higher-need neighborhoods received more service. To answer the final equity question, the next steps are to:

1. Prepare ACS tract-level indicators for transit need.
2. Prepare Montgomery County Census tract boundaries.
3. Spatially aggregate stop-level service changes to tracts.
4. Compare service changes across high-need and lower-need tracts.
5. Identify high-need places where scheduled service remains limited or declined.

This approach also reflects one of the project's smart-city critiques. The workflow turns transit equity into measurable indicators, which is useful for detecting spatial pattern. However, it should not be treated as a full account of rider experience or mobility justice.